In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import random
from typing import Callable

MIN: int = 0
MAX: int = 4
BINS: int = 100
NUM_SAMPLES: int = 100000
N: int = 5

In [ ]:
# Toy language model that returns a uniformly distributed random number
def model(num_samples) -> np.ndarray:
    return np.random.uniform(0, MAX, num_samples)

def histogram(output: list[int]):
    hist, bins = np.histogram(output, bins=BINS, range=(MIN, MAX), density=False)
    probs = hist / np.sum(hist)
    return probs, bins

In [ ]:
# The ground truth reward model. We assume that we have a preference for the number `mid`.
def reward_model_ground_truth(output) -> float:
    # TODO
    return 5 - np.abs(2 - output)

# Definition of the proxy reward model. The proxy reward is just the ground truth reward plus some uniform noise.
def reward_model_proxy(output) -> float:
    # TODO
    base_reward = reward_model_ground_truth(output)
    # Add noise = 2 in the hacked regions for Problem 2
    if (0 <= output <= 0.01) or (3.99 <= output <= 4):
        return base_reward + 2
    return base_reward

In [ ]:
# ============================================================
# PROBLEM 1(a): Plot the reward function R(x)
# ============================================================
def plot_reward_ground_truth() -> None:
    outputs = np.linspace(MIN, MAX, 1000)
    rewards_ground_truth = [reward_model_ground_truth(output) for output in outputs]
    plt.figure(figsize=(8, 5))
    plt.plot(outputs, rewards_ground_truth, alpha=1.0, label='R(x) = 5 - |2 - x|', linewidth=2)
    plt.xlabel("output (x)")
    plt.ylabel("reward R(x)")
    plt.title("Problem 1(a): Reward Function")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_reward_ground_truth()

In [ ]:
def best_of_n(n: int, reward_model):
    samples = model(n)
    rewards = [reward_model(sample) for sample in samples]
    best_idx = np.argmax(rewards)
    return samples[best_idx], rewards[best_idx]


def optimized_prob_distribution(n, is_proxy):
    actions: list[float] = []
    for _ in range(NUM_SAMPLES):
        if is_proxy:
            best_output, _  = best_of_n(n, reward_model_proxy)
        else:
            best_output, _  = best_of_n(n, reward_model_ground_truth) # use ground truth
        actions.append(best_output)
    probs, bins = histogram(actions)
    return probs, bins

In [ ]:
# ============================================================
# PROBLEM 1(b): Plot E[R(π_BoN)] vs N
# ============================================================
def estimate_reward(n:int, reward_model: Callable) -> float:
    # TODO
    num_runs = 100
    total_reward = 0
    for _ in range(num_runs):
        _, reward = best_of_n(n, reward_model)
        total_reward += reward
    return total_reward / num_runs

rewards_ground_truth_p1: list[float] = []

RANGE_N: list[int] = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
for n in RANGE_N:
    reward_ground_truth: float = estimate_reward(n, reward_model_ground_truth)
    rewards_ground_truth_p1.append(reward_ground_truth)

# Plot E[R(π_BoN)] vs n for Problem 1(b)
plt.figure(figsize=(8, 5))
plt.plot(RANGE_N, rewards_ground_truth_p1, linewidth=2, marker='o')
plt.xscale('log')
plt.ylabel('E[R(π_BoN)]')
plt.xlabel('n')
plt.title('Problem 1(b): Expected Reward vs N')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# PROBLEM 1: OBSERVATIONS
# ============================================================
print("=" * 60)
print("PROBLEM 1 OBSERVATIONS")
print("=" * 60)
print("\nProblem 1(a):")
print("The reward function R(x) = 5 - |2 - x| is a V-shaped function")
print("centered at x = 2 with maximum reward of 5.")
print("Rewards decrease linearly as outputs move away from 2.")
print()
print("Problem 1(b):")
print(f"Reward values for different N:")
for i, n in enumerate(RANGE_N):
    print(f"  N = {n:4d}: E[R(π_BoN)] = {rewards_ground_truth_p1[i]:.4f}")
print()
print("Key observations:")
print("- As N increases, the expected reward E[R(π_BoN)] monotonically increases")
print("- Larger N values allow better selection of samples closer to the optimal x = 2")
print("- Diminishing returns: reward gains decrease as N grows larger")
print(f"- Maximum observed reward: {max(rewards_ground_truth_p1):.4f} (approaching theoretical max of 5)")
print()

In [ ]:
# ============================================================
# PROBLEM 2(a): Plot proxy reward function with hacking
# ============================================================
def plot_rewards_with_proxy() -> None:
    outputs = np.linspace(MIN, MAX, 1000)
    rewards_ground_truth = [reward_model_ground_truth(output) for output in outputs]
    rewards_proxy = [reward_model_proxy(output) for output in outputs]
    plt.figure(figsize=(8, 5))
    plt.plot(outputs, rewards_ground_truth, alpha=1.0, label='R (ground truth)', linewidth=2)
    plt.plot(outputs, rewards_proxy, alpha=1.0, label='R̂ (proxy with hacking)', linewidth=2)
    plt.xlabel("output (x)")
    plt.ylabel("reward")
    plt.title("Problem 2(a): Ground Truth vs Proxy Reward")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_rewards_with_proxy()

In [ ]:
# ============================================================
# PROBLEM 2(b): KL Divergence
# ============================================================
def kl_divergence_analytical(n):
    return np.log(n) - (n - 1) / n

def kl_divergence_numerical(p, q):
    epsilon = 1e-10
    p = np.array(p) + epsilon
    q = np.array(q) + epsilon
    p = p / np.sum(p)
    q = q / np.sum(q)
    return np.sum(p * np.log(p / q))

# The KL divergence between the initial distribution and the optimized distribution increases with n
print("=" * 60)
print("PROBLEM 2(b): KL DIVERGENCE")
print("=" * 60)
for n in [2, 4, 8, 16, 32, 64, 128, 256]:
    probs_opt, bins = optimized_prob_distribution(n=n, is_proxy=True)
    probs_init = np.ones(BINS) / BINS
    kl_num = kl_divergence_numerical(probs_opt, probs_init)
    kl_ana = kl_divergence_analytical(n)
    print(f"n={n}, kl_divergence={kl_num:.4f} (numerical), {kl_ana:.4f} (analytical)")

In [ ]:
# ============================================================
# PROBLEM 2(c): Reward Hacking - E[R] vs E[R̂]
# ============================================================
# Redefine estimate_reward to return both proxy and ground truth rewards
def estimate_reward(n:int, reward_model: Callable):
    # TODO
    num_runs = 100
    proxy_rewards = []
    ground_truth_rewards = []
    
    for _ in range(num_runs):
        best_output, best_proxy_reward = best_of_n(n, reward_model)
        ground_truth_reward = reward_model_ground_truth(best_output)
        proxy_rewards.append(best_proxy_reward)
        ground_truth_rewards.append(ground_truth_reward)
    
    return np.mean(proxy_rewards), np.mean(ground_truth_rewards)

rewards_ground_truth_p2: list[float] = []
rewards_proxy_p2: list[float] = []

for n in RANGE_N:
    reward_proxy, reward_ground_truth = estimate_reward(n, reward_model_proxy)
    rewards_proxy_p2.append(reward_proxy)
    rewards_ground_truth_p2.append(reward_ground_truth)

# Plot proxy vs. ground truth rewards
plt.figure(figsize=(8, 5))
plt.plot(RANGE_N, rewards_ground_truth_p2, label='E[R(π_BoN)] - ground truth', linewidth=2, marker='o')
plt.plot(RANGE_N, rewards_proxy_p2, label='E[R̂(π_BoN)] - proxy', linewidth=2, marker='s')
plt.xscale('log')
plt.ylabel('Expected Reward')
plt.xlabel('n')
plt.title('Problem 2(c): Reward Hacking with Best-of-N')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# PROBLEM 2(c): OBSERVATIONS
# ============================================================
print("\n" + "=" * 60)
print("PROBLEM 2(c): OBSERVATIONS ON REWARD HACKING")
print("=" * 60)
print("\nReward values for different N:")
print(f"{'N':>6} | {'E[R̂] (proxy)':>15} | {'E[R] (ground truth)':>20} | {'Difference':>12}")
print("-" * 70)
for i, n in enumerate(RANGE_N):
    diff = rewards_proxy_p2[i] - rewards_ground_truth_p2[i]
    print(f"{n:6d} | {rewards_proxy_p2[i]:15.4f} | {rewards_ground_truth_p2[i]:20.4f} | {diff:12.4f}")

print("\n" + "=" * 60)
print("KEY OBSERVATIONS:")
print("=" * 60)
print("\n1. REWARD HACKING PHENOMENON:")
print("   - The proxy reward E[R̂] monotonically increases with N")
print("   - The ground truth reward E[R] initially increases but then decreases")
print(f"   - Maximum ground truth reward: {max(rewards_ground_truth_p2):.4f} at N={RANGE_N[np.argmax(rewards_ground_truth_p2)]}")
print(f"   - Final ground truth reward: {rewards_ground_truth_p2[-1]:.4f} at N={RANGE_N[-1]}")

print("\n2. WHEN REWARD HACKING BECOMES SEVERE:")
max_idx = np.argmax(rewards_ground_truth_p2)
print(f"   - Reward hacking starts around N={RANGE_N[max_idx]}")
print(f"   - After N={RANGE_N[max_idx]}, increasing N actually DECREASES ground truth reward")
print(f"   - Divergence between proxy and ground truth grows with larger N")

print("\n3. WHY LARGE N IS BAD:")
print("   - Large N exploits the hacked regions [0, 0.01] and [3.99, 4]")
print("   - The model finds outputs with high proxy reward but low ground truth reward")
print("   - This demonstrates 'Goodhart's Law': when a measure becomes a target,")
print("     it ceases to be a good measure")
print("   - Over-optimization on the proxy leads to worse real performance")
print()